# Sweep — `feature/mask-rcnn` postprocess на `best.pt`

Не учим. Коммит `c949b21`: быстрый PQ + сетка score/min_area/dedup.

GPU T4, Internet on.
Add Data: competition + Output `dm1triyaleksandrov/notebookead6e030d5`.
Add-ons → Secrets: `GITHUB_TOKEN` и `DAGSHUB_TOKEN` на **этот** ноутбук.

После sweep — полный val лучшим конфигом и CSV, если PQ ≥ 0.20. Результат пишется в DagsHub (MLflow).

In [1]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Session options → GPU T4, Restart session"

CUDA: True Tesla T4


In [2]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import os

token = ""
dagshub_token = ""
try:
    sec = UserSecretsClient()
    for name in ("GITHUB_TOKEN", "GH_TOKEN", "GIT_TOKEN", "github_token"):
        try:
            value = sec.get_secret(name)
        except Exception:
            value = ""
        if value:
            token = value
            break
    try:
        dagshub_token = sec.get_secret("DAGSHUB_TOKEN") or ""
    except Exception:
        dagshub_token = ""
except Exception as e:
    print("secrets:", type(e).__name__, e)

os.environ["GITHUB_TOKEN"] = token
if dagshub_token:
    os.environ["DAGSHUB_TOKEN"] = dagshub_token
if not token:
    raise SystemExit("Add-ons → Secrets → GITHUB_TOKEN на ЭТОТ ноутбук")
if not dagshub_token:
    raise SystemExit("Add-ons → Secrets → DAGSHUB_TOKEN на ЭТОТ ноутбук")

askpass = """#!/bin/sh
case "$1" in
    *Username*) echo "x-access-token" ;;
    *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
Path("/kaggle/working/git-askpass.sh").write_text(askpass)
os.chmod("/kaggle/working/git-askpass.sh", 0o700)

REPO = Path("/kaggle/working/repo")
if not (REPO / "run.py").exists():
    !GIT_ASKPASS=/kaggle/working/git-askpass.sh GIT_TERMINAL_PROMPT=0 \
      git clone --branch feature/mask-rcnn --single-branch \
      https://github.com/d1maal/filament-segmentation-2026.git /kaggle/working/repo
else:
    !GIT_ASKPASS=/kaggle/working/git-askpass.sh GIT_TERMINAL_PROMPT=0 \
      git -C /kaggle/working/repo fetch origin feature/mask-rcnn
    !git -C /kaggle/working/repo checkout -B feature/mask-rcnn origin/feature/mask-rcnn

%cd /kaggle/working/repo
!git log -1 --oneline
assert Path("src/models/mask_rcnn/sweep_postprocess.py").exists(), "нужен коммит c949b21+"

env_lines = Path(".env.example").read_text().splitlines()
out_lines = []
for line in env_lines:
    if line.startswith("DAGSHUB_TOKEN="):
        out_lines.append(f"DAGSHUB_TOKEN={dagshub_token}")
    elif line.startswith("DAGSHUB_EXPERIMENT_NAME="):
        out_lines.append("DAGSHUB_EXPERIMENT_NAME=mask_rcnn_postprocess")
    else:
        out_lines.append(line)
Path(".env").write_text("\n".join(out_lines) + "\n")
os.environ["DAGSHUB_EXPERIMENT_NAME"] = "mask_rcnn_postprocess"
os.environ["DAGSHUB_REPO_OWNER"] = "BraveCactus"
os.environ["DAGSHUB_REPO_NAME"] = "solar-filament-segmentation"
print("DagsHub .env ready → https://dagshub.com/BraveCactus/solar-filament-segmentation")

%pip install -q pycocotools python-dotenv tqdm mlflow dagshub matplotlib

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 211 (delta 120), reused 142 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 99.93 KiB | 2.22 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/kaggle/working/repo
c949b21 (HEAD -> feature/mask-rcnn, origin/feature/mask-rcnn) optimize PQ metrics and add Mask R-CNN postprocessing sweep
DagsHub .env ready → https://dagshub.com/BraveCactus/solar-filament-segmentation
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

**DAGSHUB / MLflow** — эта ячейка создаёт прогон. Секрет `DAGSHUB_TOKEN` должен быть прикреплён.

In [3]:
import dagshub
import mlflow

dagshub.init(
    repo_owner="BraveCactus",
    repo_name="solar-filament-segmentation",
    mlflow=True,
)
mlflow.set_experiment("mask_rcnn_postprocess")
run = mlflow.start_run(run_name="mask-rcnn-postprocess-sweep")
mlflow.log_param("model", "mask_rcnn")
mlflow.log_param("branch", "feature/mask-rcnn")
print("run_id:", run.info.run_id)
print("experiment:", mlflow.get_experiment(run.info.experiment_id).name)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7261f3fe-a214-486e-b83d-a9030c55c312&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=57179474529cfc67cb224a40159d4d49b3bf64e7d02c940fc1ac3bcb3eb55e0a




Accessing as d1maal

Initialized MLflow to track repo "BraveCactus/solar-filament-segmentation"

Repository BraveCactus/solar-filament-segmentation initialized!

2026/09/29 12:44:07 INFO mlflow.tracking.fluent: Experiment with name 'mask_rcnn_postprocess' does not exist. Creating a new experiment.


run_id: 0115c64462a14e8fb72ae201e31d63b3
experiment: mask_rcnn_postprocess


**SWEEP** — один forward на кадр, ~30 конфигов. Не `src.eval` (там mlflow).

In [4]:
from pathlib import Path

import mlflow

from src.models.mask_rcnn.sweep_postprocess import run_sweep

WEIGHTS = Path("/kaggle/working/outputs/mask_rcnn/best.pt")
if not WEIGHTS.exists():
    hits = sorted(Path("/kaggle/input").rglob("best.pt"))
    assert hits, "Add Input → Output прогона notebookead6e030d5"
    WEIGHTS = hits[0]

OUT = Path("/kaggle/working/outputs/mask_rcnn")
OUT.mkdir(parents=True, exist_ok=True)
print("weights:", WEIGHTS)
mlflow.log_param("weights", str(WEIGHTS))

results = run_sweep(
    WEIGHTS,
    limit=0,
    out_csv=OUT / "postprocess_sweep.csv",
)
best_cfg = results[0]
print("BEST CFG:", best_cfg)
mlflow.log_params({f"best_{k}": v for k, v in best_cfg.items() if k != "pq"})
mlflow.log_metrics({"sweep_best_pq": float(best_cfg["pq"])})
mlflow.log_artifact(str(OUT / "postprocess_sweep.csv"))

weights: /kaggle/input/models/dm1triyaleksandrov/mask-rcnn/pytorch/default/1/best.pt
loading annotations into memory...
Done (t=0.63s)
creating index...
index created!


sweep val: 100%|██████████| 387/387 [1:55:04<00:00, 17.84s/it]



TOP CONFIGS
01. PQ=0.3447 score=0.50 area=150 dedup=True nms=0.60 TP=1712 FP=1799 FN=1121 splits=258 merges=111
02. PQ=0.3434 score=0.50 area=150 dedup=True nms=0.70 TP=1712 FP=1825 FN=1121 splits=276 merges=111
03. PQ=0.3432 score=0.60 area=250 dedup=False nms=0.60 TP=1734 FP=1901 FN=1099 splits=556 merges=148
04. PQ=0.3428 score=0.60 area=80 dedup=False nms=0.60 TP=1734 FP=1908 FN=1099 splits=556 merges=148
05. PQ=0.3428 score=0.60 area=150 dedup=False nms=0.60 TP=1734 FP=1908 FN=1099 splits=556 merges=148
06. PQ=0.3404 score=0.45 area=80 dedup=True nms=0.60 TP=1734 FP=1937 FN=1099 splits=280 merges=117
07. PQ=0.3404 score=0.45 area=150 dedup=True nms=0.60 TP=1734 FP=1937 FN=1099 splits=280 merges=117
08. PQ=0.3389 score=0.45 area=80 dedup=True nms=0.70 TP=1734 FP=1968 FN=1099 splits=300 merges=117
09. PQ=0.3389 score=0.45 area=150 dedup=True nms=0.70 TP=1734 FP=1968 FN=1099 splits=300 merges=117
10. PQ=0.3376 score=0.40 area=80 dedup=True nms=0.60 TP=1760 FP=2058 FN=1073 splits=293

**FULL VAL + CSV** лучшим конфигом. Сабмит только если PQ ≥ 0.20.

In [5]:
import csv
import json
from pathlib import Path

import cv2
import mlflow
import numpy as np
import torch
from tqdm import tqdm

from src.config import config
from src.data import (
    ensure_val_split,
    image_stem,
    instances_for_image,
    load_coco,
    mask_to_rle_counts,
    resolve_image,
    three_channel,
)
from src.metrics import aggregate, panoptic_quality
from src.models.mask_rcnn.infer import load_checkpoint, predict_masks

MIN_PQ_FOR_SUBMIT = 0.20
OUT = Path("/kaggle/working/outputs/mask_rcnn")
OUT.mkdir(parents=True, exist_ok=True)


def val_items():
    coco = load_coco()
    val = ensure_val_split(coco)
    items = []
    for info in coco.imgs.values():
        if image_stem(info["file_name"]) not in val:
            continue
        path = resolve_image(config.train_images, info["file_name"])
        if path is not None:
            items.append((coco, info, path))
    items.sort(key=lambda item: image_stem(item[1]["file_name"]))
    return items


def to_tensor(gray):
    rgb = three_channel(gray)
    return torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).float() / 255.0


model, device = load_checkpoint(WEIGHTS)
items = val_items()
rows = []
print(
    f"full val {len(items)} | score={best_cfg['score_threshold']} "
    f"area={best_cfg['min_area']} dedup={best_cfg['deduplicate']}",
    flush=True,
)
for coco, info, path in tqdm(items, desc="val"):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise SystemExit(f"Failed to read {path}")
    pred = predict_masks(
        model,
        to_tensor(gray),
        device,
        score_threshold=best_cfg["score_threshold"],
        min_area=best_cfg["min_area"],
        deduplicate=best_cfg["deduplicate"],
        mask_nms_iou=best_cfg["mask_nms_iou"],
        containment_threshold=best_cfg["containment_threshold"],
    )
    metrics = panoptic_quality(instances_for_image(coco, info["id"]), pred)
    metrics["stem"] = Path(info["file_name"]).stem
    rows.append(metrics)

summary = aggregate(rows)
print("Home PQ full val:", round(summary["pq"], 4), summary)
(OUT / "val_metrics.json").write_text(
    json.dumps({"weights": str(WEIGHTS), "cfg": best_cfg, "summary": summary, "per_image": rows}, indent=2),
    encoding="utf-8",
)
mlflow.log_metrics(
    {
        "val_pq": float(summary["pq"]),
        "val_tp": int(summary.get("tp", 0)),
        "val_fp": int(summary.get("fp", 0)),
        "val_fn": int(summary.get("fn", 0)),
        "val_splits": int(summary.get("splits", 0)),
        "val_merges": int(summary.get("merges", 0)),
        "val_images": int(summary.get("images", 0)),
    }
)
mlflow.log_artifact(str(OUT / "val_metrics.json"))

SUBMIT = summary["pq"] >= MIN_PQ_FOR_SUBMIT
print("submit:", SUBMIT, f"(threshold {MIN_PQ_FOR_SUBMIT})")
if SUBMIT:
    csv_rows = []
    test_images = sorted(
        p for p in config.test_images.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    print(f"test: {len(test_images)}", flush=True)
    for path in tqdm(test_images, desc="test"):
        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            raise SystemExit(f"Failed to read {path}")
        pred = predict_masks(
            model,
            to_tensor(gray),
            device,
            score_threshold=best_cfg["score_threshold"],
            min_area=best_cfg["min_area"],
            deduplicate=best_cfg["deduplicate"],
            mask_nms_iou=best_cfg["mask_nms_iou"],
            containment_threshold=best_cfg["containment_threshold"],
        )
        for i, mask in enumerate(pred, start=1):
            csv_rows.append(
                {"filament_id": f"{path.stem}_{i}", "segmentation_rle": mask_to_rle_counts(mask)}
            )
    sub = OUT / "submission_mask_rcnn.csv"
    with sub.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, ["filament_id", "segmentation_rle"])
        w.writeheader()
        w.writerows(csv_rows)
    print(f"Wrote {sub} ({len(csv_rows)} rows)")
    mlflow.log_artifact(str(sub))
    !ls -lh /kaggle/working/outputs/mask_rcnn/

mlflow.end_run()
print("DagsHub run closed")

loading annotations into memory...
Done (t=0.40s)
creating index...
index created!
full val 387 | score=0.5 area=150 dedup=True


val: 100%|██████████| 387/387 [07:46<00:00,  1.21s/it]


Home PQ full val: 0.3447 {'pq': 0.3447241081856729, 'tp': 1712, 'fp': 1799, 'fn': 1121, 'splits': 258, 'merges': 111, 'images': 387}
submit: True (threshold 0.2)
test: 180


test: 100%|██████████| 180/180 [02:47<00:00,  1.07it/s]


Wrote /kaggle/working/outputs/mask_rcnn/submission_mask_rcnn.csv (1548 rows)
total 452K
-rw-r--r-- 1 root root 2.2K Sep 29 14:39 postprocess_sweep.csv
-rw-r--r-- 1 root root 359K Sep 29 14:50 submission_mask_rcnn.csv
-rw-r--r-- 1 root root  87K Sep 29 14:47 val_metrics.json
🏃 View run mask-rcnn-postprocess-sweep at: https://dagshub.com/BraveCactus/solar-filament-segmentation.mlflow/#/experiments/2/runs/0115c64462a14e8fb72ae201e31d63b3
🧪 View experiment at: https://dagshub.com/BraveCactus/solar-filament-segmentation.mlflow/#/experiments/2
DagsHub run closed
